[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/NN_DL/blob/main/NN_DL_ch02/NN_DL_ch02_BackpropViz/NN_DL_ch02_BackpropViz.ipynb)

# NN_DL_ch02_BackpropViz

**Backpropagation from Scratch with Gradient Flow Visualization**

Implement forward and backward pass of a 2-layer MLP using only NumPy. Visualize gradient flow and the vanishing gradient problem.

In [ ]:
!pip install -q matplotlib

In [ ]:
# Style & Reproducibility
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

MAIN_BLUE = '#1A3A6E'
IDA_RED   = '#CD0000'
FOREST    = '#2E7D32'
CRIMSON   = '#DC3545'
AMBER     = '#FFC107'

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3,
    'font.size': 12, 'axes.labelsize': 13, 'axes.titlesize': 14,
    'figure.figsize': (10, 6),
})
np.random.seed(42)

def save_fig(name):
    plt.savefig(f'{name}.pdf', bbox_inches='tight', dpi=300, transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', dpi=150, transparent=True)
    plt.show()
    print(f'Saved {name}.pdf and {name}.png')

In [ ]:
# Backpropagation from Scratch (NumPy)

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1 - s)

# Network: 2 -> 4 -> 4 -> 1
np.random.seed(42)
W1 = np.random.randn(2, 4) * 0.5
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 4) * 0.5
b2 = np.zeros((1, 4))
W3 = np.random.randn(4, 1) * 0.5
b3 = np.zeros((1, 1))

# XOR data
X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=np.float64)
y = np.array([[0],[1],[1],[0]], dtype=np.float64)

losses = []
grad_norms = {'W1': [], 'W2': [], 'W3': []}

lr = 0.5
for epoch in range(3000):
    # Forward
    z1 = X @ W1 + b1;  a1 = sigmoid(z1)
    z2 = a1 @ W2 + b2; a2 = sigmoid(z2)
    z3 = a2 @ W3 + b3; a3 = sigmoid(z3)

    # Loss (BCE)
    eps = 1e-8
    loss = -np.mean(y * np.log(a3 + eps) + (1 - y) * np.log(1 - a3 + eps))
    losses.append(loss)

    # Backward
    m = X.shape[0]
    dz3 = (a3 - y) / m
    dW3 = a2.T @ dz3
    db3 = np.sum(dz3, axis=0, keepdims=True)

    da2 = dz3 @ W3.T
    dz2 = da2 * sigmoid_deriv(z2)
    dW2 = a1.T @ dz2
    db2 = np.sum(dz2, axis=0, keepdims=True)

    da1 = dz2 @ W2.T
    dz1 = da1 * sigmoid_deriv(z1)
    dW1 = X.T @ dz1
    db1 = np.sum(dz1, axis=0, keepdims=True)

    grad_norms['W1'].append(np.linalg.norm(dW1))
    grad_norms['W2'].append(np.linalg.norm(dW2))
    grad_norms['W3'].append(np.linalg.norm(dW3))

    W3 -= lr * dW3; b3 -= lr * db3
    W2 -= lr * dW2; b2 -= lr * db2
    W1 -= lr * dW1; b1 -= lr * db1

    if (epoch + 1) % 500 == 0:
        print(f"Epoch {epoch+1}: Loss={loss:.4f}")

print(f"\nFinal predictions: {a3.ravel()}")
print(f"Targets:           {y.ravel()}")

In [ ]:
# Gradient Flow Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(losses, color=MAIN_BLUE, lw=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss (Backprop from Scratch)', fontweight='bold')

for (name, norms), color in zip(grad_norms.items(), [MAIN_BLUE, FOREST, IDA_RED]):
    axes[1].plot(norms, color=color, lw=1.5, label=name, alpha=0.8)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Gradient Norm')
axes[1].set_title('Gradient Norms per Layer', fontweight='bold')
axes[1].legend(); axes[1].set_yscale('log')

ratios = [n1 / (n3 + 1e-10) for n1, n3 in zip(grad_norms['W1'], grad_norms['W3'])]
axes[2].plot(ratios, color=CRIMSON, lw=1.5)
axes[2].axhline(1.0, color='grey', ls='--', lw=1, label='Equal gradients')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('||grad W1|| / ||grad W3||')
axes[2].set_title('Gradient Ratio (Layer 1 / Layer 3)', fontweight='bold')
axes[2].legend()

plt.suptitle('Backpropagation - Gradient Flow Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('nn_ch02_backprop_flow')

In [ ]:
# Vanishing Gradient Demonstration
depths = [3, 5, 10, 20]
fig, axes = plt.subplots(1, len(depths), figsize=(18, 5))

for ax, depth in zip(axes, depths):
    np.random.seed(42)
    dim = 8
    weights = [np.random.randn(2 if i == 0 else dim, dim) * 0.5 for i in range(depth)]
    weights.append(np.random.randn(dim, 1) * 0.5)

    # Forward
    activations_list = [X]
    pre_acts = []
    a = X
    for W in weights:
        z = a @ W
        pre_acts.append(z)
        a = sigmoid(z)
        activations_list.append(a)

    # Backward
    layer_grad_norms = []
    delta = (activations_list[-1] - y) / X.shape[0]
    for i in range(len(weights) - 1, -1, -1):
        grad_W = activations_list[i].T @ delta
        layer_grad_norms.append(np.linalg.norm(grad_W))
        if i > 0:
            delta = (delta @ weights[i].T) * sigmoid_deriv(pre_acts[i-1])

    layer_grad_norms.reverse()
    ax.bar(range(len(layer_grad_norms)), layer_grad_norms, color=MAIN_BLUE, edgecolor='white')
    ax.set_xlabel('Layer'); ax.set_ylabel('||grad||')
    ax.set_title(f'Depth = {depth}', fontweight='bold')
    ax.set_yscale('log')

plt.suptitle('Vanishing Gradient Problem (Sigmoid Activations)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('nn_ch02_vanishing_gradient')

## Summary

Implemented backpropagation from scratch using NumPy on a 2-layer MLP for XOR. Visualized gradient flow showing how gradients shrink exponentially in deeper sigmoid networks (vanishing gradient problem).